# 평가 분석기

며칠 또는 몇 주가 걸리던 AI 에이전트 평가 분석을 몇 분으로 단축합니다.

**문제:** 대규모 LLM-as-a-Judge 평가는 수백 개의 점수와 설명 쌍을 생성합니다. 패턴을 찾기 위해 이를 모두 읽는 작업은 시간이 많이 들고 오류가 발생하기 쉬우며 확장하기 어렵습니다.

**해결 방법:** 이 Notebook은 AI로 AI 평가 결과를 분석하고, 평가 설명을 사용해 구조적인 문제를 식별하며, 평가 대상 AI 에이전트의 system prompt를 구체적으로 개선하는 방안을 생성합니다. 개발자는 제안된 prompt를 적용해 AI 에이전트를 수동 또는 자동으로 업데이트하고, 제안된 수정 사항에 따른 개선 효과를 테스트해야 합니다.

## 활용 단계

AI 에이전트 평가는 테스트 사례로 에이전트를 테스트하고, GroundTruth 또는 LLM 기반 컨텍스트 평가를 통해 응답을 분석하고, 평가 결과를 검토한 뒤 에이전트를 개선하는 연속적인 반복 과정입니다. 에이전트를 프로덕션에 출시할 성공 기준을 충족할 때까지 이 과정을 반복합니다.

<p align="center">
<img src="assets/improvement_loop.svg" alt="지속적 개선 주기" width="700">
</p>

## 작동 방식

[Strands Agents SDK](https://github.com/strands-agents/sdk-python)를 사용해 구성합니다.

- **Orchestrator**가 점수가 낮은 모든 평가와 system prompt를 받습니다.
- 각 배치에 대해 `analyze_batch()` 도구를 호출해 **Batch Analyzer** 하위 에이전트를 실행합니다.
- **Batch Analyzer**가 LLM 판정자의 설명을 읽고 실패를 패턴별로 그룹화한 뒤 근거 인용문을 추출합니다.
- Orchestrator가 여러 배치의 패턴을 집계하고 빈도 × 심각도 순으로 정렬합니다.
- 가장 중요한 문제 3개와 바로 붙여 넣을 수 있는 prompt 수정안을 포함한 최종 보고서를 생성합니다.

<p align="center">
<img src="assets/architecture.svg" alt="아키텍처 다이어그램" width="700">
</p>

## 입력 소스

- [Strands 평가 출력](https://github.com/strands-agents/strands-evals)
- [AWS AgentCore Evaluation 출력](https://docs.aws.amazon.com/agentcore/)

## 1단계: 구성

In [ ]:
# 구성
EVAL_FOLDER = "eval_data/"
BATCH_SIZE = 10
SCORE_THRESHOLD = 0.7
SYSTEM_PROMPT_FILE = "system_prompt.txt"
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
AWS_REGION = "us-east-1"

# system prompt 로드
from pathlib import Path

prompt_path = Path(SYSTEM_PROMPT_FILE)
if not prompt_path.exists() or prompt_path.stat().st_size < 100:
    print(f"Warning: {SYSTEM_PROMPT_FILE} not found, using example_system_prompt.txt")
    SYSTEM_PROMPT_FILE = "example_system_prompt.txt"

with open(SYSTEM_PROMPT_FILE, "r") as f:
    AGENT_SYSTEM_PROMPT = f.read()

# 주석 헤더가 있으면 제거
lines = AGENT_SYSTEM_PROMPT.split("\n")
content_lines = []
in_header = True
for line in lines:
    if in_header and line.startswith("#"):
        continue
    in_header = False
    content_lines.append(line)
AGENT_SYSTEM_PROMPT = "\n".join(content_lines).strip()

## 2단계: 설정

In [ ]:
import json
import time
from pathlib import Path
from typing import List, Dict, Any, Optional
from statistics import mean, stdev
from collections import defaultdict
from IPython.display import display, Markdown

from strands import Agent, tool
from strands.models import BedrockModel

In [ ]:
def extract_evaluations(data: Any, parent_metadata: Optional[Dict] = None) -> List[Dict]:
    """임의의 JSON 구조에서 평가 결과를 재귀적으로 추출합니다."""
    evaluations = []
    parent_metadata = parent_metadata or {}

    if isinstance(data, list):
        for item in data:
            evaluations.extend(extract_evaluations(item, parent_metadata))
    elif isinstance(data, dict):
        score_key = "score" if "score" in data else ("value" if "value" in data else None)

        if score_key and "explanation" in data:
            eval_entry = {
                "score": data[score_key],
                "explanation": data["explanation"],
                "metadata": {**parent_metadata},
            }
            for key, val in data.items():
                if key not in ["score", "value", "explanation"]:
                    if isinstance(val, (str, int, float, bool)):
                        eval_entry["metadata"][key] = val
            evaluations.append(eval_entry)
        else:
            nested_metadata = {**parent_metadata}
            for key in ["session_id", "trace_id", "evaluator_name", "evaluator_id"]:
                if key in data:
                    nested_metadata[key] = data[key]
            for key in ["results", "evaluations", "data", "items"]:
                if key in data:
                    evaluations.extend(extract_evaluations(data[key], nested_metadata))

    return evaluations


def load_evaluations(folder_path: str) -> List[Dict]:
    """폴더의 모든 JSON 파일을 불러와 평가 결과를 추출합니다."""
    evaluations = []
    folder = Path(folder_path)

    for json_file in sorted(folder.glob("*.json")):
        try:
            with open(json_file, "r") as f:
                data = json.load(f)
            evals = extract_evaluations(data, {"source_file": json_file.name})
            evaluations.extend(evals)
            print(f"  {json_file.name}: {len(evals)} evaluations")
        except Exception as e:
            print(f"  {json_file.name}: Error - {e}")

    return evaluations


def compute_statistics(evaluations: List[Dict], threshold: float) -> Dict:
    """평가 결과의 통계를 계산합니다."""
    if not evaluations:
        return {"total": 0, "error": "No evaluations found"}

    scores = [e["score"] for e in evaluations if e["score"] is not None]

    by_evaluator = defaultdict(list)
    for e in evaluations:
        evaluator = e["metadata"].get("evaluator_name", e["metadata"].get("label", "unknown"))
        if e["score"] is not None:
            by_evaluator[evaluator].append(e["score"])

    evaluator_stats = {}
    for evaluator, eval_scores in by_evaluator.items():
        evaluator_stats[evaluator] = {
            "count": len(eval_scores),
            "mean": round(mean(eval_scores), 3),
            "min": round(min(eval_scores), 3),
            "max": round(max(eval_scores), 3),
        }
        if len(eval_scores) > 1:
            evaluator_stats[evaluator]["stdev"] = round(stdev(eval_scores), 3)

    low_scoring = [e for e in evaluations if e["score"] is not None and e["score"] < threshold]

    return {
        "total": len(evaluations),
        "valid_scores": len(scores),
        "mean_score": round(mean(scores), 3) if scores else None,
        "min_score": round(min(scores), 3) if scores else None,
        "max_score": round(max(scores), 3) if scores else None,
        "stdev": round(stdev(scores), 3) if len(scores) > 1 else None,
        "low_scoring_count": len(low_scoring),
        "low_scoring_pct": round(len(low_scoring) / len(scores) * 100, 1) if scores else 0,
        "by_evaluator": evaluator_stats,
        "threshold": threshold,
    }


def batch_evaluations(evaluations: List[Dict], batch_size: int) -> List[List[Dict]]:
    """평가 결과를 배치로 나눕니다."""
    return [evaluations[i : i + batch_size] for i in range(0, len(evaluations), batch_size)]

## 3단계: 데이터 로드 및 검증

In [ ]:
# 구성 검증
if not AGENT_SYSTEM_PROMPT.strip():
    raise ValueError(f"System prompt is empty. Edit {SYSTEM_PROMPT_FILE}")

if "example_system_prompt" in SYSTEM_PROMPT_FILE:
    print("Using example system prompt. Edit system_prompt.txt for best results.")
else:
    print(f"System prompt loaded ({len(AGENT_SYSTEM_PROMPT)} chars)")

eval_path = Path(EVAL_FOLDER)
if not eval_path.exists():
    raise ValueError(f"Evaluation folder not found: {EVAL_FOLDER}")

json_files = list(eval_path.glob("*.json"))
if not json_files:
    raise ValueError(f"No JSON files found in {EVAL_FOLDER}")
print(f"Found {len(json_files)} JSON files in {EVAL_FOLDER}")

In [ ]:
print(f"Loading evaluations from {EVAL_FOLDER}:")
evaluations = load_evaluations(EVAL_FOLDER)
print(f"\nTotal: {len(evaluations)} evaluations")

In [ ]:
stats = compute_statistics(evaluations, SCORE_THRESHOLD)

print(f"Mean score: {stats['mean_score']} (range: {stats['min_score']} - {stats['max_score']})")
print(f"Low scoring (<{SCORE_THRESHOLD}): {stats['low_scoring_count']} ({stats['low_scoring_pct']}%)")

print("\nBy evaluator:")
for evaluator, eval_stats in stats["by_evaluator"].items():
    print(f"  {evaluator}: mean={eval_stats['mean']}, count={eval_stats['count']}")

low_scoring = [e for e in evaluations if e["score"] is not None and e["score"] < SCORE_THRESHOLD]
print(f"\n{len(low_scoring)} low-scoring evaluations will be analyzed")

## 4단계: 분석 에이전트 정의

이 단계에서는 [Strands Agents SDK](https://github.com/strands-agents/sdk-python)를 사용해 두 에이전트를 생성합니다.

**Orchestrator**(기본 에이전트)<br>
**Batch Analyzer**(하위 에이전트)

**흐름:** `Orchestrator` → `analyze_batch()` 도구 → `Batch Analyzer` → JSON 패턴 → Orchestrator 종합 → `Report` → `사용자가 평가 대상 AI 에이전트의 System Prompt 업데이트`

In [ ]:
# ============================================
# 에이전트 PROMPT
# ============================================

BATCH_ANALYZER_PROMPT = """
You analyze low-scoring evaluations to identify systematic failure patterns.

## Input
A batch of evaluations, each with:
- score: numeric 0-1 (scores < 0.7 indicate problems)
- explanation: detailed text from the LLM judge explaining why the score was given
- metadata: context including evaluator_name, trace_id/session_id

## Your Task
1. Read each explanation carefully - these contain the LLM judge's reasoning
2. Identify SYSTEMATIC PATTERNS (not isolated incidents) in why scores are low
3. Group similar failures together
4. Extract 2-3 specific quotes as evidence per pattern
5. Note which evaluator metrics are affected

## Output (JSON)
Return ONLY valid JSON with this structure:
{
  "patterns": [
    {
      "name": "short descriptive name",
      "description": "what the agent did wrong and why it's problematic",
      "count": N,
      "evaluators_affected": ["Faithfulness", "Correctness"],
      "evidence": ["direct quote from explanation 1", "direct quote from explanation 2"],
      "root_cause": "what's missing or unclear in the system prompt"
    }
  ]
}

CONSTRAINTS:
- Maximum 5 patterns per batch
- Only include patterns that appear 2+ times (systematic, not isolated)
- Evidence quotes should be verbatim from explanations
- Root cause should identify what prompt guidance would fix this
"""

ORCHESTRATOR_PROMPT = """
You synthesize evaluation analysis into actionable recommendations with system prompt improvements.

## Your Task
1. Use the analyze_batch tool to analyze low-scoring evaluations in batches
2. Collect patterns from all batches
3. Identify the TOP 3 most impactful problems based on:
   - **Frequency**: Appears across multiple evaluations (the more, the worse)
   - **Severity**: Lower scores indicate more severe problems
   - **Fixability**: Can be addressed by clarifying the system prompt
4. Generate specific, minimal prompt changes to fix each problem

## Required Output Format

# Evaluation Analysis Report

## Summary
[2-3 sentences on overall health of the agent based on the statistics and patterns found]

## Top 3 Problems

### Problem 1: [Specific Descriptive Name]

**Evidence from evaluations:**
- "[Direct quote from LLM judge explanation]"
- "[Another direct quote showing this pattern]"

**Frequency & Impact:**
- Appears in X out of Y low-scoring evaluations
- Affects metrics: [list evaluator names]
- Average score when this occurs: X.XX

**Root Cause:**
[What's missing or unclear in the current system prompt that causes this behavior]

**Proposed Fix:**
[Specific text to add/modify in the prompt and why it will work]

---

### Problem 2: [Specific Descriptive Name]

**Evidence from evaluations:**
- "[Direct quote]"
- "[Another quote]"
- "[TraceID and SessionID]"

**Frequency & Impact:**
- Appears in X out of Y low-scoring evaluations
- Affects metrics: [list]
- Average score when this occurs: X.XX

**Root Cause:**
[What's missing in the prompt]

**Proposed Fix:**
[Specific change and rationale]

---

### Problem 3: [Specific Descriptive Name]

**Evidence from evaluations:**
- "[Direct quote]"
- "[Another quote]"

**Frequency & Impact:**
- Appears in X out of Y low-scoring evaluations
- Affects metrics: [list]
- Average score when this occurs: X.XX

**Root Cause:**
[What's missing in the prompt]

**Proposed Fix:**
[Specific change and rationale]

---

## Suggested System Prompt Changes

### Changes Summary
| # | What Changed | Original Text | New Text | Fixes |
|---|--------------|---------------|----------|-------|
| 1 | [brief description] | [exact original snippet] | [exact new snippet] | Problem 1 |
| 2 | [brief description] | [exact original snippet] | [exact new snippet] | Problem 2 |
| 3 | [brief description] | [exact original snippet] | [exact new snippet] | Problem 3 |

### Complete Updated System Prompt
```
[FULL UPDATED PROMPT - COPY-PASTE READY]
```

## CONSTRAINTS
- Only 3 problems, ranked by impact (frequency × severity)
- Evidence must be actual quotes from the evaluation explanations with traceID and sessionID
- Make minimal, surgical prompt changes (not a complete rewrite)
- Preserve everything in the original prompt that works well
- The Complete Updated System Prompt must be the FULL prompt, ready to use
- No implementation roadmaps, KPIs, timelines, or risk assessments
"""

In [ ]:
model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)

batch_analyzer_agent = Agent(model=model, system_prompt=BATCH_ANALYZER_PROMPT)


@tool
def analyze_batch(batch_json: str) -> str:
    """Analyze a batch of low-scoring evaluations to identify failure patterns."""
    result = batch_analyzer_agent(f"Analyze these evaluations and return JSON patterns:\n{batch_json}")
    return str(result)


orchestrator = Agent(model=model, system_prompt=ORCHESTRATOR_PROMPT, tools=[analyze_batch])

---

## 5단계: 분석 실행

Orchestrator는 다음 작업을 수행합니다.
1. 하위 에이전트를 사용해 평가를 배치 단위로 처리
2. 가장 중요한 문제 3개 식별
3. 구체적인 prompt 개선안 생성
4. 업데이트된 전체 system prompt 출력

In [ ]:
print(f"Analyzing {len(low_scoring)} evaluations in {len(batch_evaluations(low_scoring, BATCH_SIZE))} batches...")

batches = batch_evaluations(low_scoring, BATCH_SIZE)
batches_json = [json.dumps(batch, indent=2) for batch in batches]

analysis_prompt = f"""
Analyze these evaluation results and provide a comprehensive report.

## Statistics
- Total evaluations: {stats["total"]}
- Mean score: {stats["mean_score"]}
- Low scoring (<{SCORE_THRESHOLD}): {stats["low_scoring_count"]} ({stats["low_scoring_pct"]}%)
- Score range: {stats["min_score"]} - {stats["max_score"]}

## Evaluator Breakdown
{json.dumps(stats["by_evaluator"], indent=2)}

## Current Agent System Prompt (to be improved)
{AGENT_SYSTEM_PROMPT}

## Low-Scoring Evaluations
There are {len(batches)} batches of evaluations to analyze.
Use the analyze_batch tool for each batch:

"""

for i, batch_json in enumerate(batches_json):
    analysis_prompt += f"\nBatch {i + 1}:\n{batch_json}\n"

start_time = time.time()
result = orchestrator(analysis_prompt)
elapsed_time = round(time.time() - start_time, 2)
result_text = str(result)

print(f"Analysis complete ({elapsed_time}s)")

## 6단계: 결과 확인

이 Notebook을 실행하면 다음 결과를 확인할 수 있습니다.

- LLM 판정자의 근거 인용문이 포함된 **주요 문제 3개**
- 각 문제의 **근본 원인 분석**
- 정확한 prompt 수정 사항을 보여 주는 **변경 전후 표**
- 바로 붙여 넣을 수 있는 **업데이트된 전체 system prompt**

샘플은 [example_agent_output.md](example_agent_output.md)를 참조하세요.

In [ ]:
# ============================================
# 결과 표시
# ============================================

display(Markdown(result_text))

## 7단계: 결과 저장

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"analysis_report_{timestamp}.md"

report_content = f"""# Evaluation Analysis Report

**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
**Evaluations Analyzed:** {len(low_scoring)} low-scoring out of {stats["total"]} total
**Processing Time:** {elapsed_time}s

---

{result_text}
"""

with open(output_file, "w") as f:
    f.write(report_content)

print(f"Report saved to {output_file}")

---

## 다음 단계

분석을 실행한 후 다음 단계를 진행합니다.

1. **보고서 검토** - 식별된 문제가 관찰한 내용과 일치하는지 확인합니다.
2. **업데이트된 prompt 복사** - "Complete Updated System Prompt" 섹션을 사용합니다.
3. **점진적 테스트** - 필요한 경우 변경 사항을 한 번에 하나씩 적용합니다.
4. **재평가** - 평가 모음을 다시 실행해 개선 정도를 측정합니다.
5. **반복** - 점수가 목표에 도달할 때까지 이 과정을 반복합니다.

### 팁

- 점수가 낮은 평가가 많다면 API 호출을 줄이도록 `BATCH_SIZE`를 늘리세요.
- 여러 evaluator에 영향을 주는 파급 효과가 큰 문제부터 해결하세요.
- 변경하기 전에 원본 system prompt를 백업해 두세요.